In [20]:
# OLD CODE

In [21]:
# ── Generic Imports ────────────
import numpy as np 
from PIL import Image 
import scipy.io as sio
import matplotlib.pyplot as plt

In [22]:
# ── Constants ──────────────────
RESIZE: int = 256
DATA_PATH: str = "./../data/WillowObject/WILLOW-ObjectClass/"

In [23]:
# ── Data Class ─────────────────
class NotatedImage:
    # 1. Constructor Method
    def __init__(self, img, kpts) -> None:
        self.img: Image.Image = img 
        self.kpts: np.array = kpts

    # 2. Resize Method
    def resize(self):
        self.kpts[0] *= RESIZE / self.img.size[0]
        self.kpts[1] *= RESIZE / self.img.size[1]
        self.img = self.img.resize((RESIZE, RESIZE), resample=Image.BILINEAR)
        return self

In [24]:
# ── Path Retrieval Function ──
def getting(cat: str, n: int) -> list[NotatedImage]:
    # Local Imports
    import glob # For searching files
    import os   # To remove file extension

    images: list[str] = glob.glob(f"{DATA_PATH}{cat}/*.png")
    points: list[str] = glob.glob(f"{DATA_PATH}{cat}/*.mat")
    output = []
    set_points = set(points)

    for img_file in images:
        base, _ = os.path.splitext(img_file)
        mat_file = base + ".mat"
        if mat_file in set_points:
            img = Image.open(img_file)
            kpts = np.array(sio.loadmat(mat_file)['pts_coord'])
            ni = NotatedImage(img, kpts)
            output.append(ni)
            if len(output) >= n:
                break

    return output

In [25]:
# NEW CODE

In [26]:
class Pair:
    def __init__(self, ni_a, ni_b) -> None:
        self.ni_a = ni_a 
        self.ni_b = ni_b 

In [27]:
import numpy as np
import networkx as nx
from scipy.spatial import distance_matrix
from scipy.optimize import linear_sum_assignment

In [28]:
def enhanced_spatial_matching(self):
    # 1. Create NetworkX graphs from adjacency matrices
    # Assumes your NotatedImage attribute is named 'adj_matrix'
    G1 = nx.from_numpy_array(self.ni_a.edges) 
    G2 = nx.from_numpy_array(self.ni_b.edges)

    # 2. Compute Node2Vec embeddings
    print("Computing node2vec embeddings...")
    emb_a = compute_node2vec_embeddings(G1)
    emb_b = compute_node2vec_embeddings(G2)

    # 3. Format keypoints for distance calculation
    points_a = np.column_stack((self.ni_a.kpts[0], self.ni_a.kpts[1]))
    points_b = np.column_stack((self.ni_b.kpts[0], self.ni_b.kpts[1]))

    # 4. Calculate Distance Matrices (Vectorized avoids slow double loops)
    spatial_cost = distance_matrix(points_a, points_b)
    topological_cost = distance_matrix(emb_a, emb_b)

    # 5. Combine with specified weights
    cost_matrix = (0.6 * spatial_cost) + (0.4 * topological_cost)

    # 6. Apply Hungarian Algorithm
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # 7. Generate Matching Matrix
    matching_matrix = np.zeros(cost_matrix.shape, dtype=int)
    matching_matrix[row_ind, col_ind] = 1
    
    self.match = matching_matrix
    return self

# Append it to Pair
Pair.enhanced_match = enhanced_spatial_matching

In [29]:
import random
import numpy as np
from gensim.models import Word2Vec

def compute_node2vec_embeddings(G, dimensions=64, num_walks=10, walk_length=30):
    # 1. Generate random walks
    walks = []
    nodes = list(G.nodes())
    
    for _ in range(num_walks):
        random.shuffle(nodes) # Shuffle for better randomization
        for node in nodes:
            walk = [node]
            while len(walk) < walk_length:
                neighbors = list(G.neighbors(walk[-1]))
                if not neighbors:
                    break
                walk.append(random.choice(neighbors))
                
            # Word2Vec expects sentences (lists of strings)
            walks.append([str(n) for n in walk])

    # 2. Train the Word2Vec model (Skip-gram, sg=1)
    model = Word2Vec(sentences=walks, vector_size=dimensions, window=5, min_count=1, sg=1, workers=4)

    # 3. Extract embeddings in the exact order of the graph's nodes
    embeddings = np.zeros((len(nodes), dimensions))
    for i, node in enumerate(nodes):
        embeddings[i] = model.wv[str(node)]
        
    return embeddings

In [30]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def visualize_matching_full(pa):
    img_a, img_b = pa.ni_a.img, pa.ni_b.img
    kpts_a, kpts_b = pa.ni_a.kpts, pa.ni_b.kpts
    
    # 1. Create composite canvas
    width = max(img_a.size[0], img_b.size[0])
    height = max(img_a.size[1], img_b.size[1])
    composite = Image.new('RGB', (width * 2, height))
    composite.paste(img_a, (0, 0))
    composite.paste(img_b, (width, 0)) # Offset by 'width'
    
    plt.figure(figsize=(12, 6))
    plt.imshow(composite)
    plt.axis('off')
    
    # 2. Draw Delaunay Structure
    for i, j in pa.ni_a.edges:
        plt.plot([kpts_a[0, i], kpts_a[0, j]], [kpts_a[1, i], kpts_a[1, j]], 'y-', alpha=0.5, lw=1)
        
    for i, j in pa.ni_b.edges:
        plt.plot([kpts_b[0, i] + width, kpts_b[0, j] + width], [kpts_b[1, i], kpts_b[1, j]], 'y-', alpha=0.5, lw=1)
        
    # 3. Matching Lines
    rows, cols = np.where(pa.match == 1)
    for i, j in zip(rows, cols):
        plt.plot([kpts_a[0, i], kpts_b[0, j] + width], [kpts_a[1, i], kpts_b[1, j]], 'g-', lw=1.5, alpha=0.8)
        
    # 4. Keypoints
    plt.scatter(kpts_a[0], kpts_a[1], c='w', edgecolors='k', s=40, zorder=5)
    plt.scatter(kpts_b[0] + width, kpts_b[1], c='w', edgecolors='k', s=40, zorder=5)
    
    plt.title("Graph Matching Results")
    plt.show()

# Append to Pair if needed
Pair.visualize = visualize_matching_full